In [0]:
# Define widgets before reading their values

dbutils.widgets.text('layer_type', 'bronze')
dbutils.widgets.text('source_path', '/Volumes/workspace/ecommerce/ecommerce_data')
dbutils.widgets.text('process_date', '2026-01-16')

# Read parameter values from widgets
layer_type = dbutils.widgets.get('layer_type')
source_path = dbutils.widgets.get('source_path')
process_date = dbutils.widgets.get('process_date')

print(f"Layer Type: {layer_type}")
print(f"Source Path: {source_path}")
print(f"Process Date: {process_date}")

# You can now use these variables in your ETL logic below.

In [0]:
if layer_type == "bronze":
    # Bronze layer logic
    print("Processing Bronze layer...")
    df = spark.read.format("csv").option("header", "true").load(source_path)
    display(df)
elif layer_type == "silver":
    # Silver layer logic
    print("Processing Silver layer...")
    df_bronze = spark.read.format("csv").option("header", "true").load(source_path)
    df = df_bronze.filter(df_bronze["event_time"] == process_date).dropDuplicates()
    display(df)
elif layer_type == "gold":
    # Gold layer logic
    print("Processing Gold layer...")
    df_bronze = spark.read.format("csv").option("header", "true").load(source_path)
    df_silver = df_bronze.filter(df_bronze["event_time"] == process_date).dropDuplicates()
    df = df_silver.groupBy("category_id").count()
    display(df)
else:
    print("Unknown layer type!")

In [0]:
if layer_type == "bronze":
    print("Processing Bronze layer...")
    df = spark.read.format("csv").option("header", "true").load(source_path)
    display(df)

In [0]:
# Silver Layer: Data Cleaning & Transformation
# This step reads the raw bronze data, cleans it, and prepares it for analytics.
# Why? To ensure data quality and consistency for downstream use.

if layer_type == "silver":
    print("Processing Silver layer...")
    # Read raw bronze data
    df_bronze = spark.read.format("csv").option("header", "true").load(source_path)
    # Filter by process_date using the correct column 'event_time'
    # This narrows the data to the relevant date for analysis
    df_silver = df_bronze.filter(df_bronze["event_time"] == process_date)
    # Remove duplicate rows to ensure data integrity
    df_silver = df_silver.dropDuplicates()
    # Show the cleaned and filtered data
    display(df_silver)
    # Next: Save df_silver to a Silver table or file if needed


In [0]:
# Inspect the columns of the bronze DataFrame to find the correct date column
if layer_type == "silver":
    df_bronze = spark.read.format("csv").option("header", "true").load(source_path)
    print("Bronze DataFrame columns:", df_bronze.columns)
    display(df_bronze.limit(5))  # Show a sample of the data for reference

In [0]:
# Gold Layer: Aggregation & Business Metrics
# This step aggregates the cleaned silver data to produce business insights.
# Why? To create summary tables for reporting, dashboards, or ML models.

if layer_type == "gold":
    print("Processing Gold layer...")
    # Read raw bronze data
    df_bronze = spark.read.format("csv").option("header", "true").load(source_path)
    # Clean and filter as in Silver layer, using 'event_time' for date filtering
    df_silver = df_bronze.filter(df_bronze["event_time"] == process_date).dropDuplicates()
    # Example aggregation: group by 'category_id' and count rows
    df_gold = df_silver.groupBy("category_id").count()
    # Show the aggregated results
    display(df_gold)
    # Next: Save df_gold to a Gold table or file if needed
